# Session 6 — Measuring Agent Performance

**Spine:** *Comparing two versions is flipping two coins: report the interval, not the winner — and when it crosses zero, the tie goes to the cheaper agent.*

## Today's question

Last week you built a benchmark. Today you use it to answer the question every team asks before shipping a change to an agent: **is the new version better, worse — or can't we tell?**

We compare **three versions of the same Deep Research agent**. Same model, same tools, same base prompt; each candidate adds **one line** to the prompt:

| version | what changed | role |
|---|---|---|
| **healthy** | nothing | the **incumbent** — what is in production today |
| **redundant** | + *"run every search query a second time… to confirm the result is stable"* | candidate |
| **concise** | + *"Answer in 2 sentences."* | candidate |

Each candidate is compared against healthy, on the same 12 rows. Checking a new version against the current one before you ship it is a **regression test**.

**One word you will see:** an **arm** is one version in a comparison. *"Runs per arm"* = runs of **each** version.

The deck carries the argument; this notebook carries only what you run. Every hands-on is a script in the repo — nothing is hidden, you just will not type it. The one file you **edit** is `criteria6.py`.

**Nothing today calls an agent.** Last night the instructor ran every version on every row, five times, and saved the results in `runs6.json`. You work from that file. Tavily cost today: **zero**. Only Hands-on 2 and 4 touch LangSmith, and every block after them works without it.

### Where the 180 runs come from

A **run** = one full execution of the agent on one question: model calls, searches, answer.

| | count | what it is |
|---|---|---|
| **versions** (arms) | **3** | healthy, redundant, concise |
| **rows** | **12** | one row = one question in the dataset `s5-class-benchmark-pool`, tag `v1` |
| ↳ different questions | **11** | one question — *"latest released version of `langgraph` on PyPI"* — is in the dataset **twice**, as two rows |
| **runs of each row** | **5** | the **same** version answering the **same** question again, nothing changed. Any difference between the 5 is noise |
| **total** | **3 × 12 × 5 = 180** | |

So:
- **one version = 12 rows × 5 runs = 60 runs = one experiment** in LangSmith. Three versions, three experiments.
- **one question = 5 runs per version** — except the duplicated one, which has **10**, because the analysis groups runs by question text. You will see "10 runs" on one line of the opener table; that is why.

The cell after the environment check counts all of this from the file, so you do not have to take the table on trust.

## What does "the interval" mean? (read this before anything else)

The spine says *"report the interval"* and *"when it crosses zero"*. Here is what that means.

**Everyday example.** You try a new route to college a few times. On average it is **3 minutes faster**. Is it really faster — or did you just get lucky days?

| your honest answer | includes 0 ("no difference")? | so… |
|---|---|---|
| "between **2 and 4 minutes faster**" | no | it **is** faster |
| "between **8 minutes faster and 2 minutes slower**" | **yes** | you **can't tell** |

That range is the **interval** (a *95% confidence interval*): the range the real difference probably sits in, given how noisy your trips were. **"Crosses zero"** means the range includes "no difference".

**Our agents, the same way** (token cost vs healthy, measured on last night's runs — you will recompute these yourself in Hands-on 5):

| candidate | its average said | the interval | crosses zero? | verdict |
|---|---|---|---|---|
| **redundant** | +46% | **+18% to +74%** | no | really more expensive — a regression |
| **concise** | −26% | **−75% to +23%** | **yes** | a **tie** — we can't tell |

**The rule:** range doesn't include 0 → a real difference. Range includes 0 → a tie. On a tie, pick the cheaper agent; if you can't tell which is cheaper either, keep what is in production.

*"95%" means: repeat the whole experiment many times, and ranges built this way contain the true difference about 95 times in 100. More runs → a narrower range.*

In [ ]:
!git pull

In [ ]:
# Environment check. Prints WHERE you are before importing anything, so a wrong
# kernel gives a named error instead of a spinner.
import sys, os
print("python:", sys.executable)
print("cwd   :", os.getcwd())
assert os.path.exists("runs6.json"), "runs6.json missing: are you in course-repo? did git pull work?"

from dotenv import load_dotenv
load_dotenv()          # every LangSmith call needs this. push_pool.py skipped it once; nothing got pushed.
print("LangSmith key loaded:", bool(os.environ.get("LANGSMITH_API_KEY")))

import importlib, json, statistics
import paired as P
import consistency6
runs = P.load("runs6.json")
print(f"runs6.json: {len(runs)} runs,", sorted({r['version'] for r in runs}))

In [ ]:
# Where the 180 runs come from -- counted from runs6.json, not typed in.
from collections import Counter
versions = Counter(r["version"] for r in runs)                              # arms
row_ids  = {r["example_id"] for r in runs}                                  # dataset rows
per_row  = Counter((r["version"], r["example_id"]) for r in runs)           # runs of each row
per_q    = Counter(r["question"] for r in runs if r["version"] == "healthy")
reps     = max(per_row.values())
print(f"versions (arms)     : {len(versions)}   {dict(versions)}")
print(f"rows in the dataset : {len(row_ids)}")
print(f"different questions : {len(per_q)}")
print(f"runs of each row    : {reps}   (same version, same question, run again)")
print(f"total               : {len(versions)} x {len(row_ids)} x {reps} = {len(runs)} runs")
print(f"one version         : {len(row_ids)} x {reps} = {len(row_ids) * reps} runs = one LangSmith experiment")
for q, n in per_q.items():
    if n > reps:
        print(f"\nin the dataset twice, so {n} runs per version: {q}")

## Opener — where does the noise live? *(projector)*

Session 5: *σ/µ ≈ 40%, so to detect a 10% difference between two versions you need **252 runs of each version**.* That σ came from **one** question, run four times.

Last night the **same version — healthy — answered each of the 11 questions 5 times**, with nothing changed between runs. A **run** is one full execution of the agent on one question: model calls, searches, answer. So any difference between a question's 5 runs is pure noise, not a version difference.

Before you look — which kind of question do you think moves the most?

In [ ]:
# PROJECTOR. healthy, tokens_billed: every run of each question, quietest first.
# Nothing changed between runs: same agent, same question, same model.
#   cheapest / priciest = tokens_billed of that question's cheapest and most expensive run
#   wobble (CV)         = standard deviation / mean of THAT question's own runs
import textwrap
rows = P.by_row(runs, "healthy", "tokens_billed")
table = []
for q, v in rows.items():
    m, sd = statistics.fmean(v), statistics.stdev(v)
    table.append((sd / m, q, min(v), max(v), m, len(v)))
for cv, q, lo, hi, m, n in sorted(table):
    print(textwrap.fill(q, width=100))
    print(f"    {n:>2} runs   cheapest {lo:>7,.0f}   priciest {hi:>7,.0f}   "
          f"average {m:>7,.0f}   wobble (CV) {cv:4.0%}\n")

### Reading the numbers: SD, CV and σ/µ

| symbol | name | what it tells you | unit |
|---|---|---|---|
| **σ** (sigma) | **standard deviation (SD)** | how far a typical run lands from its average | tokens |
| **µ** (mu) | **mean** | the average | tokens |
| **σ/µ** | **coefficient of variation (CV)** — the "wobble" column | the SD as a % of the average | % |

**Why both.** A phone bill of ₹3,000 that swings ±₹500 month to month, and rent of ₹25,000 that swings ±₹500. Same SD — ₹500. But the phone bill wobbles **17%** and the rent **2%**. SD alone cannot tell you whether a wobble is big; CV can.

**Use SD** when you are talking in real units (*"this question moves by ~8,000 tokens"*). **Use CV** to compare questions of different sizes, or when your claim is a percentage (*"can we see a 10% change?"*).

The cell below works it through on two questions from the table above.

In [ ]:
# Worked example: SD vs CV on two questions from the table above.
by_cv = sorted(table)                      # (cv, question, cheapest, priciest, average, runs)
for cv, q, lo, hi, m, n in (by_cv[len(by_cv) // 2], by_cv[-1]):   # the median question, the noisiest
    sd = cv * m
    print(textwrap.fill(q, width=100))
    print(f"    average (µ) {m:>7,.0f} tokens    SD (σ) {sd:>6,.0f} tokens    "
          f"CV = σ/µ = {sd:,.0f} / {m:,.0f} = {cv:.0%}\n")

In [ ]:
# PROJECTOR. The same data as three numbers, and what each costs.
# runs of EACH version needed to detect a 10% difference between two versions
# (95% confidence, 80% power): 2 * (1.96 + 0.84)^2 * CV^2 / 0.10^2
n_for = lambda cv: round(2 * (1.96 + 0.8416) ** 2 * cv ** 2 / 0.01)
cvs = sorted(t[0] for t in table)
s_all, df, m_all = P.within_row_sigma(runs, "healthy", "tokens_billed")
print(f"median question   CV {statistics.median(cvs):.2f}  -> {n_for(statistics.median(cvs)):>4} runs per version to detect a 10% difference")
print(f"noisiest question CV {cvs[-1]:.2f}  -> {n_for(cvs[-1]):>4} runs per version to detect a 10% difference")
print(f"pooled, all rows  CV {s_all / m_all:.2f}  -> {n_for(s_all / m_all):>4}   <- belongs to no question")

# Now take ONE question out -- the noisiest -- and pool the other ten again.
noisiest = max(table)[1]
s_rest, _, m_rest = P.within_row_sigma(P.without(runs, noisiest), "healthy", "tokens_billed")
print(f"\npooled CV, all {len(table)} questions:          {s_all / m_all:.2f}")
print(f"pooled CV, noisiest question removed: {s_rest / m_rest:.2f}")
print("removed:", textwrap.fill(noisiest, width=90, subsequent_indent=" " * 9))

## Hands-on 1 — your criteria, before any number *(8 min)*

Edit **`criteria6.py`** in VS Code. Three things:

1. **`SUCCESS_BAR`** — the pass rate an agent must reach to be production-ready at all.
2. **`ACT_IF`** — the smallest change you would *act on*. Not the smallest you could detect.
3. **`PREDICT`** — for `redundant` (*"run every search twice"*) and `concise` (*"Answer in 2 sentences."*): HIGHER / LOWER / TIE on each metric, versus healthy.

Then run the checker until it says OK. **Your predictions are locked when it does** — Hands-on 5 scores them.

*A threshold chosen after you have seen the result is not a threshold, it is a caption.*

In [ ]:
!python criteria6.py

## Hands-on 2 — the runs, in YOUR LangSmith *(10 min)*

```bash
python replay6.py --dry     # key + file check, touches nothing
python replay6.py           # pushes the v1 pool if you lack it, then 3 experiments
```

A **replay**: a target that returns a saved output instead of calling a model. Zero model cost, zero Tavily, three real experiments in your workspace.

**⚠ The trap.** LangSmith will show **~0.0 s latency and 0 tokens** for every replayed run — nothing was called. Read the **feedback** columns (`tokens_billed`, `n_searches`, `latency_s`), which carry what the agent recorded when it really ran.

Then in the UI: Datasets → `s5-class-benchmark-pool` → Experiments → tick two → **Compare**.

*No key, or a 401?* Skip it. Every block after this runs from `runs6.json` alone.

In [ ]:
!python replay6.py --dry

In [ ]:
!python replay6.py

## Hands-on 3 — the same question, five times *(8 min)*

```bash
python consistency6.py
```

**pass@1** is how good the agent is on average. **pass^5** is whether you can rely on it: a question that passes 4 runs in 5 has pass@1 = 0.8 and pass^5 = 0.

Answer with your partner, each with a number:
1. Which metric has the biggest gap between pass@1 and pass^5?
2. Session 5 said σ/µ ≈ 40%. For which questions is that true?

In [ ]:
!python consistency6.py

## Hands-on 4 — a benchmark becomes a regression set *(6 min · sacrificial)*

```bash
python make_regression_set.py --dry
python make_regression_set.py
```

A benchmark row says what a **right** answer looks like. A regression row also says what the **incumbent did** on it — median tokens and searches over healthy's five runs. The next candidate is scored against that row, by a rule: *within 25% of what production does today, on the same question.*

In [ ]:
!python make_regression_set.py --dry

In [ ]:
!python make_regression_set.py

## Hands-on 5 — the regression test *(12 min)* ⚑ THE SESSION

```bash
python regress6.py
```

For each candidate, each metric: the change, its **95% interval** (the range the real difference probably sits in — see the top of this notebook), and what it means against **your** `ACT_IF` — TIE · REAL, BELOW BAR · REAL, MAYBE BAR · CLEARS BAR. Then your predictions, scored. Then the decision, by the tie rule.

In [ ]:
!python regress6.py

In [ ]:
# PROJECTOR. concise vs healthy, tokens, question by question.
import textwrap
pcts = P.row_pcts(runs, "healthy", "concise", "tokens_billed")
for q, v in sorted(pcts.items(), key=lambda kv: kv[1]):
    print(f"{round(v):+5d}%   " + textwrap.fill(q, width=90, subsequent_indent=" " * 9))
# The four numbers to read out. sgn() prints 0 as "0", not "+0" or "-0".
sgn = lambda x: f"{round(x):+d}" if round(x) else "0"
big_q, big_v = max(pcts.items(), key=lambda kv: abs(kv[1]))          # the question that moved most
rest = [v for q, v in pcts.items() if q != big_q]
lines = [("1. mean of the paired difference:", f"{sgn(P.paired(runs, 'healthy', 'concise', 'tokens_billed').pct)}%   <- the headline"),
         ("2. median question:", f"{sgn(statistics.median(pcts.values()))}%"),
         ("3. the biggest single question:", f"{sgn(big_v)}%   " + textwrap.shorten(big_q, 60)),
         (f"4. the other {len(rest)} questions:", f"{sum(abs(v) <= 3 for v in rest)} of {len(rest)} within ±3%")]
print()
for label, value in lines:
    print(f"{label:<36}{value}")

## Automating it *(instructor demo · sacrificial)*

A regression test someone has to remember to run stops being run. `regress_gate.py` is the same arithmetic reduced to what CI understands: an **exit code**.

In [ ]:
!python regress_gate.py --cand redundant; echo "exit code: $?"
!python regress_gate.py --cand concise;   echo "exit code: $?"

In [ ]:
# Why the gate pairs runs itself instead of using LangSmith's evaluate_comparative.
# Measured by preflight6, 10 Sep: with num_repetitions=5 the comparator received
# this many runs per example. The docs describe a two-item list.
print(json.load(open("deck_numbers6.json")).get("comparative_runs_per_example"))

## Hands-on 6 — the report, and your recommendation *(8 min)*

```bash
python report6.py
```

It writes `report6_<you>.md`. Everything is generated **except section 5**: one sentence, by hand, and it **must quote an interval**.

- Not allowed: *"concise is 26% cheaper."*
- Allowed: *"concise's token change is −26%, 95% interval −75% to +23%, so we keep healthy."*

This is the shape of the mid-term.

In [ ]:
!python report6.py